# Organoid quality and blacklist candidate review

**Role:** Analysis.

Inspect validation errors and measured/predicted curvature on individual organoids using a restored model. Rank candidates for manual data-quality review, inspect their meshes, and export a draft list if requested. A poorly predicted organoid is not automatically bad data. The notebook never changes training splits or applies a new blacklist, and it does not retrain outcome-selected models.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/models/gnn.py").is_file())  # Locate the repository from the notebook’s working directory.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Repository root used to resolve data and result paths.

## Select model and candidate settings

Load the saved validation cohort and choose how many organoids to inspect. Draft exports are separate from the dataset blacklist.

In [ ]:
import json, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from src.artifacts.runs import AnalysisRun
from src.analysis.metrics.evaluation import prediction_table
from src.artifacts.paths import resolve_training_run
TRAINING_NOTEBOOK = 'gin_depth_training'  # Folder under training_results containing the saved run.
TRAINING_RUN = None  # Set a run directory name; None selects only if exactly one exists.
RUN_DIR = resolve_training_run(ROOT, TRAINING_NOTEBOOK, TRAINING_RUN)  # Also accepts an explicit historical path.
run = AnalysisRun(RUN_DIR)
display(run.records)

# Saved model selection
MODEL_KEY = run.records.iloc[0].key  # Checkpoint key selected from the saved model catalog.

# Inference resources
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # Use CUDA when available, otherwise CPU.
selection = run.select(MODEL_KEY, device=DEVICE)
model = selection['model']
marker_names = selection['marker_names']
g_train, g_val = selection['groups']['train'], selection['groups']['val']
OUTPUT_DIR = RUN_DIR/'analysis'/'cohort_review'/MODEL_KEY  # Directory for this analysis’s tables and figures.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Example selection
TOP_N = 12  # Number of highest-error organoids displayed for review.

# Exports
EXPORT_DRAFT = False  # Export review candidates without changing the blacklist.

## Rank validation organoids

Compute physical-curvature MSE, bias and extreme residuals. Examine count and collection metadata alongside these scores before interpreting any candidate as a quality problem.

In [ ]:
cells=prediction_table(selection,device=DEVICE)
ranked=cells.groupby('organoid_str').agg(n_cells=('node','size'),mse=('squared_error','mean'),
    bias=('error','mean'),max_absolute_error=('error',lambda x:np.abs(x).max())).sort_values('mse',ascending=False)
display(ranked.head(TOP_N))
ranked.to_csv(OUTPUT_DIR/'organoid_diagnostics.csv')
fig,axes=plt.subplots(1,2,figsize=(10,4))
axes[0].scatter(ranked.n_cells,ranked.mse);axes[0].set(xlabel='Cell count',ylabel='MSE')
axes[1].hist(cells.error,bins=80);axes[1].set(xlabel='Prediction error',ylabel='Cells');plt.show()

## Inspect a candidate mesh

Project the same node-level truth and prediction onto the saved mesh. Missing mesh metadata is reported for manual inspection rather than silently selecting another organoid.

In [ ]:
from src.plotting.mesh_plots import project_node_quantities_to_mesh, plot_projected_true_vs_pred
CANDIDATE = ranked.index[0]  # Organoid selected for manual quality inspection.
raw=selection['raw_graphs'][CANDIDATE]
frame=cells[cells.organoid_str==CANDIDATE].sort_values('node')
projection=project_node_quantities_to_mesh(raw,node_true=frame.y_true.to_numpy(),node_pred=frame.y_pred.to_numpy())
fig=plot_projected_true_vs_pred(projection)
fig.show()

## Export a review draft

Save candidate IDs and metrics for review only. This is not an approved exclusion list and is not written into training data.

In [ ]:
if EXPORT_DRAFT:
    ranked.head(TOP_N).to_csv(OUTPUT_DIR/'blacklist_candidates_review_only.csv')